In [2]:
import csv
f = open("knapsack.csv")
csvfile = csv.DictReader(f, delimiter='\t')

headers = csvfile.fieldnames

table = []
for row in csvfile:
    table.append(row)
    
f.close()

In [3]:
print(headers)

['Project', 'Required Programmers', 'Required Capital', 'Estimated NPV']


In [4]:
# read the parameters
ReqProgrammers = {}
ReqCapital = {}
EstNPV = {}
for row in table:
    if row['Project'] != 'Available':
        ReqProgrammers[row['Project']] = float(row['Required Programmers'])
        ReqCapital[row['Project']] = float(row['Required Capital'])
        EstNPV[row['Project']] = float(row['Estimated NPV'])
Budget = float(table[-1]['Required Capital'])
AvailableProgrammers = float(table[-1]['Required Programmers'])

In [5]:
EstNPV

{'1': 650.0,
 '2': 550.0,
 '3': 600.0,
 '4': 450.0,
 '5': 375.0,
 '6': 525.0,
 '7': 750.0}

In [6]:
from docplex.mp.model import Model
mdl = Model()

Intel MKL WARNING: Support of Intel(R) Streaming SIMD Extensions 4.2 (Intel(R) SSE4.2) enabled only processors has been deprecated. Intel oneAPI Math Kernel Library 2025.0 will require Intel(R) Advanced Vector Extensions (Intel(R) AVX) instructions.
Intel MKL WARNING: Support of Intel(R) Streaming SIMD Extensions 4.2 (Intel(R) SSE4.2) enabled only processors has been deprecated. Intel oneAPI Math Kernel Library 2025.0 will require Intel(R) Advanced Vector Extensions (Intel(R) AVX) instructions.


In [7]:
# variables
select = mdl.binary_var_dict(EstNPV, name='project')

In [8]:
select

{'1': docplex.mp.Var(type=B,name='project_1'),
 '2': docplex.mp.Var(type=B,name='project_2'),
 '3': docplex.mp.Var(type=B,name='project_3'),
 '4': docplex.mp.Var(type=B,name='project_4'),
 '5': docplex.mp.Var(type=B,name='project_5'),
 '6': docplex.mp.Var(type=B,name='project_6'),
 '7': docplex.mp.Var(type=B,name='project_7')}

In [6]:
# objective
mdl.maximize(mdl.sum(EstNPV[i]*select[i] for i in select))

In [7]:
# constraints
MaxProgrammers = mdl.add_constraint(mdl.sum(ReqProgrammers[i]*select[i] for i in select) <= AvailableProgrammers)
MaxBudget = mdl.add_constraint(mdl.sum(ReqCapital[i]*select[i] for i in select) <= Budget)

In [8]:
# solve
mdl.solve()
mdl.get_solve_details()

docplex.mp.SolveDetails(time=0.030967,status='integer optimal solution')

In [9]:
mdl.print_solution()

objective: 1925.000
status: OPTIMAL_SOLUTION(2)
  project_1=1
  project_6=1
  project_7=1


In [10]:
mdl.add_kpi(MaxProgrammers.lhs, 'Programmers used')
mdl.add_kpi(MaxBudget.lhs, 'Budget used')

DecisionKPI(name=Budget used,expr=250project_1+175project_2+300project_3+150project_4+145project_5..)

In [11]:
mdl.report()

* model docplex_model1 solved with objective = 1925.000
*  KPI: Programmers used = 19.000
*  KPI: Budget used      = 735.000


In [12]:
# Add logical constraints based on the generic rule.
# We wish to forbid these combinations:
# (x1, x2, x3) = (1,0,0), (0,1,0), and (0,0,1)
# and no other combinations!
Rules = { 'R1', 'R2', 'R3' }
Combination = { 
    ('R1','1') : 1, ('R1','2') : 0, ('R1','3') : 0,
    ('R2','1') : 0, ('R2','2') : 1, ('R2','3') : 0,
    ('R3','1') : 0, ('R3','2') : 0, ('R3','3') : 1
}

In [13]:
RuleConstraints = {}
for r in Rules:
    # sum the 1s
    SumOnes = mdl.sum(select[i] for i in select if (r,i) in Combination if Combination[r,i] == 1)
    # sum the 0s
    SumZeroes = mdl.sum((1-select[i]) for i in select if (r,i) in Combination if Combination[r,i] == 0)
    RuleConstraints[r] = mdl.add_constraint(SumOnes + SumZeroes <= 2)

In [14]:
for r in Rules:
    print(RuleConstraints[r])

-project_1+project_2-project_3+2 <= 2
project_1-project_2-project_3+2 <= 2
-project_1-project_2+project_3+2 <= 2


In [15]:
# solve
mdl.solve()
mdl.get_solve_details()

docplex.mp.SolveDetails(time=0.020587,status='integer optimal solution')

In [16]:
mdl.print_solution()

objective: 1775.000
status: OPTIMAL_SOLUTION(2)
  project_1=1
  project_3=1
  project_6=1
